# Product entity resolution on WDC Products: full experiment notebook

One notebook, top to bottom, produces every number, table and figure for the paper.
Run **Runtime → Run all** on a GPU runtime (T4 is enough; L4/A100 is faster).

**What it measures**

| # | Question | Output |
|---|---|---|
| A | Matching quality of eight matchers across four architecture families: token-set, word co-occurrence + SVM, GBDT + attribute features (no deep learning); fine-tuned bi-encoder; fine-tuned RoBERTa and DeBERTa-v3 cross-encoders; two zero-shot LLM families (Qwen2.5, Phi-3.5). All nine WDC Products variants | `table_matching` |
| B | Does choosing the threshold **on validation** for *entity-level* F1 beat the usual pairwise-F1 threshold? (no test-set tuning anywhere) | `table_entity` |
| C | Hard attribute veto: AP change, F1 at a shared threshold, and **veto precision measured at the base matcher's own operating point**, with bootstrap CIs | `table_constraint` |
| D | Is veto precision a function of base-matcher quality? (one point per matcher × variant × seed) | `fig_veto_vs_quality` |
| E | Same conflict evidence used softly (recalibration feature) instead of as a veto | `table_constraint` |
| F | **Conflict-aware review routing**: at a fixed reviewer budget, does routing by conflict-aware error probability fix more errors than plain uncertainty? | `table_review`, `fig_review` |
| G | Robustness to unseen products, calibration (ECE) | `table_robustness`, `fig_reliability` |
| H | Do the pairwise findings (C, D, E, F) hold on a second benchmark family (Abt-Buy, Amazon-Google, Walmart-Amazon)? | `table_generalisation` |
| I | Cost of each matcher (minutes per run, ms per pair) | `table_cost` |
| ✓ | **Paper-readiness check** against success criteria fixed in the config cell before running | `PAPER_READINESS.md` |

**How it is organised**

* **Stage A (GPU)** scores every validation and test pair once per matcher and seed, and caches the scores
  (`scores/*.csv.gz`). Cached runs are skipped, so after a disconnect you just run all cells again.
  Train/validation splits are shared by the three unseen levels, so each model trains **once per
  corner-case level**, not once per variant.
* **Stage B (CPU)** reads only the cached scores. It can be re-run anywhere, including a laptop.

**Runtime (L4, full config):** roughly 7-8 h in total: 18 RoBERTa and 18 DeBERTa fits (6 datasets/levels x 3
seeds), 18 bi-encoder fits, two LLMs, and about 15 min of CPU work. Every finished piece is cached on Drive,
so the run can span several sessions. To save compute, remove "deberta" or "llm2" from `MATCHERS` before
starting (never after seeing results).
Set `QUICK = True` first for a 10-15 minute end-to-end check.

**Protocol rules enforced in code:** models are fit on train only. Temperatures, thresholds, ceilings, cleanup
gates and the review recalibrator are fit on validation only. Test labels are used only for evaluation, and
the one test-tuned number (`oracle`) is labelled as an upper bound.

In [ ]:
# ===================== 1. CONFIGURATION (edit here only) =====================
QUICK = False            # True = fast end-to-end smoke run (one variant, tiny models)

RUN_NAME   = "full_v1"   # output folder name; change it to start a fresh run
CORNERS    = [80, 50, 20]
UNSEEN     = [0, 50, 100]
DEV_SIZE   = "medium"
SEEDS      = [0, 1, 2]
TRAIN_SUBSAMPLE = 0.9    # each seed refits on a different 90% resample of train (real variance)
MATCHERS   = ["tokenset", "wordcooc", "gbdt", "biencoder", "deberta", "crossencoder", "llm", "llm2"]
# Architectures, chosen BEFORE running and all reported (no picking the one that "works"):
#   tokenset / wordcooc / gbdt  - no-ML, linear and tree baselines (CPU)
#   biencoder    - fine-tuned Siamese encoder: each offer encoded separately, cosine -> probability
#   crossencoder - fine-tuned RoBERTa-base cross-encoder: the PRIMARY strong matcher (has published numbers)
#   deberta      - fine-tuned DeBERTa-v3-base cross-encoder: independent replication of the strong matcher
#   llm / llm2   - zero-shot instruction-tuned LLMs from two different model families
BI_MODEL, BI_EPOCHS, BI_MAX_LEN = "sentence-transformers/all-mpnet-base-v2", 5, 128
DEBERTA_MODEL = "microsoft/deberta-v3-base"
# Second-benchmark family (Magellan/DeepMatcher data in Ditto format). Pairwise analyses only: these
# benchmarks ship no gold clusters, so entity-level results come from WDC Products alone.
EXTRA_DATASETS = ["abt_buy", "amazon_google", "walmart_amazon"]

# Success criteria, fixed BEFORE running. Section 11 checks the results against them.
# Do not edit them after seeing results: that would be p-hacking, and reviewers look for it.
CRITERIA = {"min_wins_frac": 7 / 9, "alpha": 0.05, "rho_alpha": 0.01, "budget": 0.05}

# Cross-encoder (Ditto-style serialisation, as in the WDC Products paper)
CE_MODEL, CE_EPOCHS, CE_BATCH, CE_LR, CE_MAX_LEN, CE_MAX_STEPS = "roberta-base", 8, 32, 2e-5, 256, None

# Zero-shot LLM, scored as P(Yes)/(P(Yes)+P(No)) from one forward pass (no generation).
# T4: Qwen2.5-3B-Instruct. L4/A100: "Qwen/Qwen2.5-7B-Instruct" also fits.
LLM_MODEL, LLM_BATCH, LLM_MAX_CHARS = "Qwen/Qwen2.5-3B-Instruct", 16, 250
LLM2_MODEL = "microsoft/Phi-3.5-mini-instruct"   # second, unrelated LLM family (MIT licence, not gated)
LLM_CORNERS   = None     # None = all CORNERS
LLM_MAX_PAIRS = None     # cap pairs per split (None = all); only for quick checks

MIN_RECALL     = 0.5     # precision-forward rule (kept for comparability with earlier tables)
KAPPA_GRID     = [0.0, 0.05, 0.15, 0.25, 0.35, 0.45]   # veto ceilings, chosen on validation AP
GATES          = [None, 0.4, 0.6, 0.8]                  # transitive-cleanup gates (None = no cleanup)
REVIEW_BUDGETS = [0.01, 0.02, 0.05, 0.10]               # share of test pairs a reviewer checks
N_BOOT         = 1000    # paired Poisson bootstrap resamples over test pairs
USE_DRIVE      = True    # Colab: keep outputs on Google Drive so disconnects lose nothing
AUTO_SCALE     = True    # on a >=22 GB GPU (Colab Pro L4/A100) switch the LLM to Qwen2.5-7B-Instruct
GITHUB_URL     = "https://github.com/Vikramsrini/MajorProject.git"

# Published F1, medium development set: Peeters, Der & Bizer, "WDC Products", EDBT 2024, Table 3
# (docs/base_paper/). Columns seen / half-seen / unseen = 0 / 50 / 100% unseen. "crossencoder" = their
# RoBERTa-base row, so the comparison is like-for-like only while CE_MODEL = "roberta-base".
PUBLISHED_F1 = {
    "wordcooc": {(80, 0): .5266, (80, 50): .4406, (80, 100): .3057, (50, 0): .5807, (50, 50): .4604,
                 (50, 100): .2970, (20, 0): .5804, (20, 50): .5133, (20, 100): .3438},
    "crossencoder": {(80, 0): .7218, (80, 50): .7205, (80, 100): .7013, (50, 0): .7858, (50, 50): .7591,
                     (50, 100): .7114, (20, 0): .8368, (20, 50): .8060, (20, 100): .7835},
}
PUBLISHED_F1_80_0 = {m: v[(80, 0)] for m, v in PUBLISHED_F1.items()}

if QUICK:
    RUN_NAME, CORNERS, UNSEEN, SEEDS = "quick", [80], [0, 100], [0]
    CE_MODEL, CE_EPOCHS, CE_MAX_STEPS = "distilroberta-base", 1, 60
    LLM_MODEL, LLM_MAX_PAIRS, N_BOOT = "Qwen/Qwen2.5-0.5B-Instruct", 300, 200
    BI_EPOCHS = 1
    EXTRA_DATASETS = ["abt_buy"]
KEYS = list(CORNERS) + list(EXTRA_DATASETS)          # every dataset/variant family scored in Stage A
UNSEEN_OF = {**{c: UNSEEN for c in CORNERS}, **{k: [0] for k in EXTRA_DATASETS}}
DATASET_NAME = {"abt_buy": "Abt-Buy", "amazon_google": "Amazon-Google", "walmart_amazon": "Walmart-Amazon"}
LLM_CORNERS = [c for c in (LLM_CORNERS or KEYS) if c in KEYS]
print("run:", RUN_NAME, "| WDC corners:", CORNERS, "| unseen:", UNSEEN, "| extra:", EXTRA_DATASETS,
      "| seeds:", SEEDS, "| matchers:", MATCHERS)

## 2. Environment, repository, output folders

In [ ]:
import os, sys, subprocess, pathlib, json, time, gc, re, shutil, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

IN_KAGGLE = os.path.isdir("/kaggle/working")
IN_COLAB = (not IN_KAGGLE) and ("google.colab" in sys.modules)
BASE = pathlib.Path("/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else ".").resolve()
PROJ = BASE / "MajorProject"

if not (PROJ / "src" / "er").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(PROJ)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJ}[finetune]"], check=True)
if str(PROJ / "src") not in sys.path:
    sys.path.insert(0, str(PROJ / "src"))

OUT = BASE / "er_runs" / RUN_NAME
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = pathlib.Path("/content/drive/MyDrive/er_runs") / RUN_NAME
    except Exception as exc:  # noqa: BLE001
        print("Drive not mounted; outputs stay in this session only:", exc)
for sub in ("data", "scores", "logs", "tables", "figures", "latex"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if HAS_GPU else "none")
GPU_MATCHERS = ("biencoder", "deberta", "crossencoder", "llm", "llm2")
if not HAS_GPU and any(m in MATCHERS for m in GPU_MATCHERS):
    print("!! No GPU: dropping the neural matchers. Use Runtime > Change runtime type > GPU for the full study.")
    MATCHERS = [m for m in MATCHERS if m not in GPU_MATCHERS]
if HAS_GPU and AUTO_SCALE and not QUICK:
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    if VRAM_GB >= 22:   # L4 = 24 GB, A100 = 40/80 GB. The cross-encoder stays roberta-base, bs 32, for comparability.
        LLM_MODEL, LLM_BATCH = "Qwen/Qwen2.5-7B-Instruct", (32 if VRAM_GB >= 40 else 16)
    print(f"GPU memory {VRAM_GB:.0f} GB -> LLM {LLM_MODEL} (batch {LLM_BATCH})")
print("outputs ->", OUT)

## 3. Data: WDC Products pair-wise benchmark (63 MB, public)

For each corner-case level this builds one table of validation pairs plus the three test splits
(0/50/100% unseen), writes it to `data/pairs_c{corner}.csv.gz`, and stores gold cluster ids in
`data/records_c{corner}.csv.gz`. Stage B needs only these files and the scores.

In [ ]:
from er.io import wdc_index, load_wdc_products_pairwise
from er.schema import Dataset, PairTable

RAW = PROJ / "data" / "raw"
if not list(RAW.rglob("*.json.gz")):
    subprocess.run([sys.executable, str(PROJ / "scripts" / "download_wdc.py"), "--dest", str(RAW)], check=True)
IDX = wdc_index(RAW)
print(len(IDX), "benchmark files")

def prepare_corner(c):
    idx = IDX[IDX.corner_cases == c]
    def path(split, unseen=None):
        sub = idx[idx.split == split]
        sub = sub[sub.unseen == unseen] if unseen is not None else sub[sub.dev_size == DEV_SIZE]
        if sub.empty:
            raise FileNotFoundError(f"no {split} file for corner={c} unseen={unseen} dev={DEV_SIZE}")
        return sub.iloc[0]["path"]
    ds_tr, tr = load_wdc_products_pairwise(path("train"))
    ds_va, va = load_wdc_products_pairwise(path("valid"))
    tests = {u: load_wdc_products_pairwise(path("test", u)) for u in UNSEEN}
    frames = [ds_tr.df, ds_va.df] + [d.df for d, _ in tests.values()]
    ds = Dataset(f"wdc{c}cc", pd.concat(frames, ignore_index=True).drop_duplicates("record_id"))
    parts = [va.labeled.frame.assign(split="valid")]
    parts += [pt.labeled.frame.assign(split=f"test_{u}") for u, (_, pt) in tests.items()]
    P = pd.concat(parts, ignore_index=True)[["split", "record_id_l", "record_id_r", "label", "is_hard_negative"]]
    P["label"] = P["label"].astype(int)
    P["is_hard_negative"] = P["is_hard_negative"].fillna(False).astype(bool).astype(int)
    P.insert(0, "pid", np.arange(len(P)))
    return ds, tr, P

CORNER = {}
for c in CORNERS:
    ds, tr, P = prepare_corner(c)
    CORNER[c] = {"ds": ds, "train": tr, "P": P}
    P.to_csv(OUT / "data" / f"pairs_c{c}.csv.gz", index=False)
    ds.df[["record_id", "gold_entity", "title"]].to_csv(OUT / "data" / f"records_c{c}.csv.gz", index=False)
    print(f"corner {c}: records={ds.n} train={len(tr.labeled.frame)} "
          + " ".join(f"{k}={v}" for k, v in P.split.value_counts().sort_index().items()))

# ---- second benchmark family: Abt-Buy, Amazon-Google, Walmart-Amazon (Ditto/Magellan format)
import hashlib, urllib.request
DITTO_BASE = "https://raw.githubusercontent.com/megagonlabs/ditto/master/data/er_magellan"
DITTO = {"abt_buy": ("Textual", "Abt-Buy"), "amazon_google": ("Structured", "Amazon-Google"),
         "walmart_amazon": ("Structured", "Walmart-Amazon")}

def parse_ditto(s):
    """'COL name VAL x COL description VAL COL price VAL 9' -> {name: x, description: '', price: 9}."""
    out = {}
    for chunk in re.split(r"(?:^|\s)COL\s+", s.strip()):
        key, sep, val = chunk.partition(" VAL")
        if not sep and chunk.endswith("VAL"):
            key, sep, val = chunk[:-3], "VAL", ""
        if sep and key.strip():
            out[key.strip()] = val.strip()
    return out

def ditto_record(attrs, source):
    title = attrs.get("title") or attrs.get("name") or ""
    brand = attrs.get("brand") or attrs.get("manufacturer") or ""
    extra = [attrs.get("description", "")] + [f"{k} {attrs[k]}" for k in ("modelno", "category") if attrs.get(k)]
    description = " ".join(x for x in extra if x)
    lead = brand if brand and brand.lower() not in title.lower() else ""
    rid = hashlib.md5(json.dumps(attrs, sort_keys=True).encode()).hexdigest()[:16]
    return rid, {"record_id": rid, "text": " ".join(x for x in (lead, title, description) if x),
                 "title": title, "description": description, "source": source, "gold_entity": None,
                 "attrs": {"brand": brand.lower()} if brand else {}}

def prepare_ditto(key):
    kind, folder = DITTO[key]
    d = RAW / "ditto" / folder
    d.mkdir(parents=True, exist_ok=True)
    recs, tables = {}, {}
    for split in ("train", "valid", "test"):
        f = d / f"{split}.txt"
        if not f.exists():
            urllib.request.urlretrieve(f"{DITTO_BASE}/{kind}/{folder}/{split}.txt", f)
        rows = []
        for line in f.read_text(encoding="utf-8").splitlines():
            parts = line.split("\t")
            if len(parts) != 3:
                continue
            ids = []
            for side in parts[:2]:
                rid, rec = ditto_record(parse_ditto(side), key)
                recs.setdefault(rid, rec)
                ids.append(rid)
            rows.append((ids[0], ids[1], int(parts[2])))
        tables[split] = (pd.DataFrame(rows, columns=["record_id_l", "record_id_r", "label"])
                         .drop_duplicates(["record_id_l", "record_id_r"]).assign(is_hard_negative=0))
    ds = Dataset(DATASET_NAME[key], pd.DataFrame(list(recs.values())))
    tr = PairTable(tables["train"].reset_index(drop=True), {"dataset": key})
    P = pd.concat([tables["valid"].assign(split="valid"), tables["test"].assign(split="test_0")], ignore_index=True)
    P = P[["split", "record_id_l", "record_id_r", "label", "is_hard_negative"]]
    P.insert(0, "pid", np.arange(len(P)))
    return ds, tr, P

for k in EXTRA_DATASETS:
    ds, tr, P = prepare_ditto(k)
    CORNER[k] = {"ds": ds, "train": tr, "P": P}
    P.to_csv(OUT / "data" / f"pairs_c{k}.csv.gz", index=False)
    ds.df[["record_id", "gold_entity", "title"]].to_csv(OUT / "data" / f"records_c{k}.csv.gz", index=False)
    print(f"{DATASET_NAME[k]}: records={ds.n} train={len(tr.frame)} valid={(P.split == 'valid').sum()} "
          f"test={(P.split == 'test_0').sum()} test positives={P[P.split == 'test_0'].label.sum()}")

# Dataset statistics for the paper's setup table
stats = []
for k, d in CORNER.items():
    P, trf = d["P"], d["train"].labeled.frame
    for split in sorted(P.split.unique()):
        if split == "valid":
            continue
        t = P[P.split == split]
        stats.append({"key": str(k), "dataset": "WDC Products" if str(k).isdigit() else DATASET_NAME[k],
                      "variant": f"{k}% cc / {split.split('_')[1]}% unseen" if str(k).isdigit() else "standard",
                      "records": d["ds"].n, "train": len(trf), "train_pos": int(trf.label.sum()),
                      "valid": int((P.split == "valid").sum()), "test": len(t), "test_pos": int(t.label.sum()),
                      "test_hard_neg": int(((t.is_hard_negative == 1) & (t.label == 0)).sum())})
pd.DataFrame(stats).to_csv(OUT / "data" / "dataset_stats.csv", index=False)

## 4. Attribute and specifier conflicts (the constraint signal)

Uses the repository's own constraint logic (`AttributeConstrainedMatcher`) so the signal is identical to
the paper's s5/s9 systems:

* `conf_full`: typed attributes, asymmetric with the truncation waiver, plus unit-numeric specifiers (= s5/s9).
* `conf_sym`: stated-vs-stated conflicts only, a higher-precision variant.

In [ ]:
from er.models.attribute import AttributeConstrainedMatcher

def conflict_table(ds, P):
    full = AttributeConstrainedMatcher(None, ds, mode="asymmetric_subset", use_specifiers=True)
    sym = AttributeConstrainedMatcher(None, ds, mode="symmetric", use_specifiers=True)
    full._ensure(); sym._ensure()
    rows = []
    for pid, l, r in zip(P.pid, P.record_id_l, P.record_id_r):
        cf, cs = full._conflicts_for(l, r), sym._conflicts_for(l, r)
        rows.append((pid, int(bool(cf)), int(bool(cs)), ";".join(sorted({x.split(":")[0] if x.startswith("specifier") else x for x in cf}))))
    return pd.DataFrame(rows, columns=["pid", "conf_full", "conf_sym", "reasons"])

for c in KEYS:
    f = OUT / "data" / f"conflicts_c{c}.csv.gz"
    if not f.exists():
        conflict_table(CORNER[c]["ds"], CORNER[c]["P"]).to_csv(f, index=False)
    C = pd.read_csv(f)
    print(f"corner {c}: conflict share full={C.conf_full.mean():.3f} sym={C.conf_sym.mean():.3f}")

## 5. Stage A: score every pair (resumable)

In [ ]:
def score_tag(m):
    """Cache key names the model, so scores from a different model are never silently reused."""
    if m == "llm":
        return "llm-" + LLM_MODEL.split("/")[-1]
    if m == "llm2":
        return "llm-" + LLM2_MODEL.split("/")[-1]
    if m == "crossencoder":
        return f"crossencoder-{CE_MODEL.split('/')[-1]}-e{CE_EPOCHS}"
    if m == "deberta":
        return f"crossencoder-{DEBERTA_MODEL.split('/')[-1]}-e{CE_EPOCHS}"
    if m == "biencoder":
        return f"biencoder-{BI_MODEL.split('/')[-1]}-e{BI_EPOCHS}"
    return m

def score_path(c, m, seed):
    return OUT / "scores" / f"scores_c{c}_{score_tag(m)}_s{seed}.csv.gz"

def save_scores(path, pids, s):
    s = np.asarray(s, dtype=float)
    assert len(s) == len(pids), "score/pair length mismatch"
    if not np.isfinite(s).all():
        raise ValueError(f"{path.name}: {int((~np.isfinite(s)).sum())} non-finite scores")
    tmp = path.with_name(path.name + ".tmp.csv.gz")
    pd.DataFrame({"pid": pids, "score": s}).to_csv(tmp, index=False)
    os.replace(tmp, path)

def train_subset(tr, seed):
    df = tr.labeled.frame
    if TRAIN_SUBSAMPLE >= 1.0:
        return tr
    n = max(2, int(round(TRAIN_SUBSAMPLE * len(df))))
    return PairTable(df.sample(n=n, random_state=seed).reset_index(drop=True), dict(tr.meta))

def seeds_for(m):
    return [0] if m in ("tokenset", "llm", "llm2") else SEEDS   # these have nothing to fit

def log_time(c, m, seed, seconds, n_pairs):
    """Cost table for the paper: wall-clock per (dataset, matcher, seed) on this hardware."""
    f = OUT / "logs" / "timing.csv"
    pd.DataFrame([{"dataset": c, "matcher": m, "model": score_tag(m), "seed": seed, "seconds": round(seconds, 1),
                   "pairs_scored": n_pairs, "hardware": torch.cuda.get_device_name(0) if HAS_GPU else "cpu"}]
                 ).to_csv(f, mode="a", header=not f.exists(), index=False)

def free_gpu():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

In [ ]:
# --- 5a. CPU matchers: token-set similarity, word co-occurrence + SVM, GBDT + attribute features
from er.models.baselines import SimilarityMatcher, WordCooccurrenceMatcher, ClassifierMatcher

def make_cpu_matcher(m, ds, seed):
    if m == "tokenset":
        return SimilarityMatcher(metric="token_set").bind(ds)
    if m == "wordcooc":
        return WordCooccurrenceMatcher(ds)
    if m == "gbdt":
        return ClassifierMatcher(ds, feature_groups={"fuzz", "tok", "num", "attr"}, random_state=seed)
    raise ValueError(m)

for c in KEYS:
    ds, tr, P = CORNER[c]["ds"], CORNER[c]["train"], CORNER[c]["P"]
    for m in [x for x in ("tokenset", "wordcooc", "gbdt") if x in MATCHERS]:
        for seed in seeds_for(m):
            out = score_path(c, m, seed)
            if out.exists():
                continue
            t0 = time.time()
            matcher = make_cpu_matcher(m, ds, seed)
            if m != "tokenset":
                matcher.fit(train_subset(tr, seed))
            save_scores(out, P.pid.values, matcher.score(P))
            log_time(c, m, seed, time.time() - t0, len(P))
            print(f"[{m}] {c} seed {seed}: {time.time() - t0:.0f}s")

In [ ]:
# --- 5b. Fine-tuned cross-encoder (one fit per corner level x seed; scores valid + all test splits)
CE_RUNS = [(m, mid) for m, mid in (("crossencoder", CE_MODEL), ("deberta", DEBERTA_MODEL)) if m in MATCHERS]
if CE_RUNS:
    from er.models.crossencoder import CrossEncoderMatcher
for m_ce, model_id in CE_RUNS:
    for c in KEYS:
        ds, tr, P = CORNER[c]["ds"], CORNER[c]["train"], CORNER[c]["P"]
        for seed in seeds_for(m_ce):
            out = score_path(c, m_ce, seed)
            if out.exists():
                print(f"[skip] {m_ce} {c} seed {seed}")
                continue
            t0 = time.time()
            ce = CrossEncoderMatcher(ds, model_name=model_id, epochs=CE_EPOCHS, batch_size=CE_BATCH, lr=CE_LR,
                                     max_len=CE_MAX_LEN, seed=seed, max_steps=CE_MAX_STEPS)
            ce.fit(train_subset(tr, seed))
            s = ce.score(P)
            save_scores(out, P.pid.values, s)
            log_time(c, m_ce, seed, time.time() - t0, len(P))
            pd.DataFrame(ce.history).to_csv(OUT / "logs" / f"{m_ce}_history_c{c}_s{seed}.csv", index=False)
            v = (P.split == "valid").values
            yv, sv = P.label.values[v], s[v]
            best_f1 = max((2 * ((sv >= t) & (yv == 1)).sum() / max(1, (sv >= t).sum() + yv.sum()))
                          for t in np.quantile(sv, np.linspace(0.5, 0.995, 40)))
            print(f"[{m_ce}] {c} seed {seed}: {(time.time() - t0) / 60:.1f} min, best valid F1 ~ {best_f1:.3f}")
            if best_f1 < 0.5 and not QUICK:
                print("  !! valid F1 below 0.5: check the training log before trusting this run")
            del ce
            free_gpu()

In [ ]:
# --- 5b'. Fine-tuned bi-encoder (Siamese): each offer is encoded on its own, so pairs never see each other.
# Same training pairs, serialisation and early-stopping rule as the cross-encoders, which makes the
# bi- vs cross-encoder comparison about architecture only.
from er.models.crossencoder import serialize_record, _max_f1_at_best_threshold

class BiEncoderMatcher:
    def __init__(self, ds, model_name, epochs=5, batch_size=32, lr=2e-5, max_len=128, seed=0, es_fraction=0.12):
        self.ds, self.model_name, self.epochs, self.bs, self.lr = ds, model_name, epochs, batch_size, lr
        self.max_len, self.seed, self.es_fraction = max_len, seed, es_fraction
        self.text = {str(r["record_id"]): serialize_record(r) for r in ds.df.to_dict("records")}

    def _encode(self, ids):
        enc = self.tok([self.text.get(i, "") for i in ids], truncation=True, max_length=self.max_len,
                       padding=True, return_tensors="pt").to(DEVICE)
        h = self.model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        v = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)            # mean pooling
        return torch.nn.functional.normalize(v.float(), dim=-1)

    def _predict(self, left, right):
        self.model.eval()
        uniq = sorted(set(left) | set(right))
        pos = {u: i for i, u in enumerate(uniq)}
        chunks = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            for i in range(0, len(uniq), 256):
                chunks.append(self._encode(uniq[i:i + 256]).cpu())
        E = torch.cat(chunks)
        cos = (E[[pos[x] for x in left]] * E[[pos[x] for x in right]]).sum(-1)
        return torch.sigmoid(self.w.detach().cpu() * cos + self.b.detach().cpu()).numpy().astype(float)

    def fit(self, train):
        from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        df = train.labeled.frame.reset_index(drop=True)
        L, Rr = df.record_id_l.astype(str).values, df.record_id_r.astype(str).values
        Y = df.label.astype(int).values
        idx = rng.permutation(len(df))
        n_es = max(1, int(round(self.es_fraction * len(df))))
        es, fit_idx = idx[:n_es], idx[n_es:]
        self.tok = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name).to(DEVICE)
        self.w = torch.nn.Parameter(torch.tensor(10.0, device=DEVICE))   # cosine -> logit scale
        self.b = torch.nn.Parameter(torch.tensor(-5.0, device=DEVICE))
        opt = torch.optim.AdamW([{"params": self.model.parameters(), "lr": self.lr},
                                 {"params": [self.w, self.b], "lr": 1e-2}])
        steps = self.epochs * int(np.ceil(len(fit_idx) / self.bs))
        sched = get_linear_schedule_with_warmup(opt, max(1, int(0.1 * steps)), steps)
        amp = DEVICE == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
        best = (-1.0, None)
        for ep in range(self.epochs):
            self.model.train()
            order = rng.permutation(fit_idx)
            for st in range(0, len(order), self.bs):
                bt = order[st:st + self.bs]
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    cos = (self._encode(list(L[bt])) * self._encode(list(Rr[bt]))).sum(-1)
                logits = self.w * cos + self.b
                loss = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits, torch.tensor(Y[bt], dtype=torch.float32, device=DEVICE))
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
            f1 = _max_f1_at_best_threshold(Y[es], self._predict(list(L[es]), list(Rr[es])))
            print(f"  epoch {ep + 1}/{self.epochs}: loss {loss.item():.4f}, early-stopping F1 {f1:.4f}")
            if f1 > best[0]:
                best = (f1, ({k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()},
                             self.w.item(), self.b.item()))
        state, w, b = best[1]
        self.model.load_state_dict(state)
        with torch.no_grad():
            self.w.fill_(w); self.b.fill_(b)
        return self

    def score(self, pairs):
        return self._predict(list(pairs.record_id_l.astype(str)), list(pairs.record_id_r.astype(str)))

if "biencoder" in MATCHERS:
    for c in KEYS:
        ds, tr, P = CORNER[c]["ds"], CORNER[c]["train"], CORNER[c]["P"]
        for seed in seeds_for("biencoder"):
            out = score_path(c, "biencoder", seed)
            if out.exists():
                print(f"[skip] biencoder {c} seed {seed}")
                continue
            t0 = time.time()
            bi = BiEncoderMatcher(ds, BI_MODEL, epochs=BI_EPOCHS, max_len=BI_MAX_LEN, seed=seed).fit(train_subset(tr, seed))
            save_scores(out, P.pid.values, bi.score(P))
            log_time(c, "biencoder", seed, time.time() - t0, len(P))
            print(f"[biencoder] {c} seed {seed}: {(time.time() - t0) / 60:.1f} min")
            del bi
            free_gpu()

In [ ]:
# --- 5c. Zero-shot LLM: P(Yes) vs P(No) from the next-token distribution
LLM_SYSTEM = "You are an expert in product data integration."
LLM_USER = ("Do the following two product offers refer to the same real-world product? "
            "Answer with Yes or No only.\n\nOffer 1: {a}\nOffer 2: {b}")

def offer_text(rec):
    t = str(rec.get("title") or "").strip() or str(rec.get("text") or "")[:200]
    attrs = rec.get("attrs")
    brand = attrs.get("brand") if isinstance(attrs, dict) else None
    if brand and str(brand).lower() not in t.lower():
        t = f"{brand} {t}"
    return re.sub(r"\s+", " ", t)[:LLM_MAX_CHARS]

def answer_token_ids(tok, words):
    """First non-whitespace token of each spelling. SentencePiece tokenizers (e.g. Phi) split ' Yes' into
    a bare space token plus 'Yes'; taking the space token would make Yes and No indistinguishable."""
    ids = set()
    for w in words:
        enc = [t for t in tok.encode(w, add_special_tokens=False) if tok.decode([t]).strip()]
        if enc:
            ids.add(enc[0])
    return ids

def chat_prompt(tok, user):
    try:
        return tok.apply_chat_template([{"role": "system", "content": LLM_SYSTEM}, {"role": "user", "content": user}],
                                       tokenize=False, add_generation_prompt=True)
    except Exception:  # noqa: BLE001 - some templates reject a system turn; fold it into the user turn
        return tok.apply_chat_template([{"role": "user", "content": LLM_SYSTEM + "\n\n" + user}],
                                       tokenize=False, add_generation_prompt=True)

def llm_scores(ds, pairs, model_name):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.padding_side, tok.truncation_side = "left", "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    text = {str(r["record_id"]): offer_text(r) for r in ds.df.to_dict("records")}
    prompts = [chat_prompt(tok, LLM_USER.format(a=text.get(l, ""), b=text.get(r, "")))
               for l, r in zip(pairs.record_id_l, pairs.record_id_r)]
    yes = answer_token_ids(tok, ["Yes", " Yes", "yes", " yes"])
    no = answer_token_ids(tok, ["No", " No", "no", " no"])
    shared = yes & no
    yes, no = sorted(yes - shared), sorted(no - shared)
    assert yes and no, f"could not find distinct Yes/No tokens for {model_name}"
    order = np.argsort([len(p) for p in prompts], kind="mergesort")
    out = np.full(len(prompts), np.nan)
    major = torch.cuda.get_device_capability()[0] if HAS_GPU else 0
    if HAS_GPU:   # bf16 needs Ampere+; T4 runs fp16. NaN scores are recomputed in fp32.
        dtypes = [torch.bfloat16 if major >= 8 else torch.float16, torch.float32]
    else:
        dtypes = [torch.float32]
    for dtype in dtypes:
        todo = order[np.isnan(out[order])]
        if len(todo) == 0:
            break
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(DEVICE).eval()
        with torch.no_grad():
            for i in range(0, len(todo), LLM_BATCH):
                idx = todo[i:i + LLM_BATCH]
                enc = tok([prompts[j] for j in idx], return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(DEVICE)
                try:
                    logits = model(**enc, logits_to_keep=1).logits[:, -1, :]
                except TypeError:
                    logits = model(**enc).logits[:, -1, :]
                lp = torch.log_softmax(logits.float(), dim=-1)
                d = torch.logsumexp(lp[:, yes], -1) - torch.logsumexp(lp[:, no], -1)
                out[idx] = torch.sigmoid(d).cpu().numpy()
        del model
        free_gpu()
        if np.isnan(out).any():
            print(f"  {int(np.isnan(out).sum())} NaN scores in {dtype}; retrying them in higher precision")
    return out

for m_llm, model_id in (("llm", LLM_MODEL), ("llm2", LLM2_MODEL)):
    if m_llm not in MATCHERS:
        continue
    for c in LLM_CORNERS:
        out = score_path(c, m_llm, 0)
        if out.exists():
            print(f"[skip] {m_llm} {c}")
            continue
        P = CORNER[c]["P"]
        sub = P if not LLM_MAX_PAIRS else pd.concat(
            [g.sample(n=min(len(g), LLM_MAX_PAIRS), random_state=0) for _, g in P.groupby("split")])
        t0 = time.time()
        s = llm_scores(CORNER[c]["ds"], sub, model_id)
        save_scores(out, sub.pid.values, s)
        log_time(c, m_llm, 0, time.time() - t0, len(sub))
        print(f"[{m_llm}] {model_id.split('/')[-1]} {c}: {len(sub)} pairs in {(time.time() - t0) / 60:.1f} min")

## 6. Stage B: analysis library (CPU only, reads cached scores)

Definitions used in every table:

* **Thresholds** are chosen in raw-score space on validation: `thr_f1` maximises validation F1 (the usual
  practice), `thr_pf` maximises precision at recall ≥ 0.5. Ties go to the strictest threshold.
* **Calibration**: Platt scaling on validation. ECE uses 10 equal-mass bins.
* **Hard veto at a shared threshold**: accept `score ≥ thr_f1` *and* no conflict. **Veto precision** is the
  share of non-matches among pairs the base matcher accepts at `thr_f1` that the veto rejects, so it counts
  only pairs whose decision the veto actually changes.
* **Capped score** (for AP): conflicting scores are capped at κ, with κ chosen on validation AP.
* **Soft constraint**: Platt recalibration with conflict features `[z, full, sym, full·z, sym·z]` fit on
  validation. Its threshold is chosen on validation F1.
* **Entity level**: connected components over accepted test edges, scored by pairwise F1 over the
  transitive closure. Configurations: `pw` = thr_f1; `ent` = threshold that maximises *validation* entity
  F1; `clean_pw` = thr_f1 plus the cleanup gate chosen on validation; `best` = (threshold, gate) chosen on
  validation; `oracle` = best on test (upper bound only, not a result).
* **Review routing**: the reviewer checks k = budget·N test pairs and corrects them. `uncertainty` ranks by
  error probability under Platt; `conflict_aware` ranks by error probability under the conflict-aware
  recalibrator; `conflict_first` reviews accepted conflicting pairs first; `random` is averaged over 10 draws.
* **CIs**: paired Poisson bootstrap over test pairs (`N_BOOT` resamples), 95% percentile intervals.

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import wilcoxon, spearmanr
from sklearn.linear_model import LogisticRegression

EPS = 1e-6

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1 - EPS)
    return np.log(p / (1 - p))

def _cut_table(y, s):
    order = np.argsort(-s, kind="mergesort")
    ys, ss = y[order], s[order]
    tp, fp = np.cumsum(ys), np.cumsum(1 - ys)
    ends = np.r_[np.nonzero(ss[1:] != ss[:-1])[0], len(ss) - 1]
    return ss[ends], tp[ends], fp[ends]

def pick_threshold(y, s, rule="f1", min_recall=MIN_RECALL):
    """Threshold on s chosen from (y, s). Cuts are tie-block boundaries; ties -> strictest cut."""
    y, s = np.asarray(y, int), np.asarray(s, float)
    thr, tp, fp = _cut_table(y, s)
    prec = tp / np.maximum(tp + fp, 1)
    rec = tp / max(1, int(y.sum()))
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    obj = f1
    if rule == "pf":
        obj = np.where(rec >= min_recall, prec, -1.0)
        if (obj < 0).all():
            obj = f1
    i = np.nonzero(obj >= obj.max() - 1e-12)[0][0]   # thresholds descend, so the first is the strictest
    return float(thr[i])

def prf(y, pred):
    y, pred = np.asarray(y, int), np.asarray(pred, bool)
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def ap_weighted(y, s, W):
    """Tie-aware average precision for each row of weights W (B, N); equals sklearn AP when W = 1."""
    y, s = np.asarray(y, float), np.asarray(s, float)
    order = np.argsort(-s, kind="mergesort")
    ys, ss, Wo = y[order], s[order], W[:, order]
    ends = np.r_[np.nonzero(ss[1:] != ss[:-1])[0], len(ss) - 1]
    blk_end = np.repeat(ends, np.diff(np.r_[-1, ends]))
    pos = Wo * ys
    tp, tot = np.cumsum(pos, axis=1), np.cumsum(Wo, axis=1)
    prec = (tp / np.maximum(tot, 1e-12))[:, blk_end]
    return (pos * prec).sum(1) / np.maximum(pos.sum(1), 1e-12)

def ap(y, s):
    return float(ap_weighted(y, s, np.ones((1, len(y))))[0])

def f1_weighted(y, pred, W):
    y, pred = np.asarray(y, int), np.asarray(pred, bool)
    tp = W @ (pred & (y == 1)).astype(float)
    fp = W @ (pred & (y == 0)).astype(float)
    fn = W @ (~pred & (y == 1)).astype(float)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)

def ci(delta_boot):
    lo, hi = np.percentile(delta_boot, [2.5, 97.5])
    return float(lo), float(hi)

def ece(p, y, n_bins=10):
    p, y = np.asarray(p, float), np.asarray(y, int)
    edges = np.quantile(p, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, len(edges) - 2)
    return float(sum((idx == b).mean() * abs(p[idx == b].mean() - y[idx == b].mean())
                     for b in range(len(edges) - 1) if (idx == b).any()))

def reliability(p, y, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    return pd.DataFrame([{"bin_mid": (edges[b] + edges[b + 1]) / 2, "count": int((idx == b).sum()),
                          "mean_p": float(p[idx == b].mean()) if (idx == b).any() else np.nan,
                          "observed": float(y[idx == b].mean()) if (idx == b).any() else np.nan}
                         for b in range(n_bins)])

class Platt:
    """Logistic recalibration of logit(score); with conf, adds conflict features and interactions."""
    def __init__(self, with_conflicts=False):
        self.with_conflicts = with_conflicts
        self.lr = LogisticRegression(C=100.0, max_iter=5000)
    def _X(self, s, conf):
        z = logit(s)[:, None]
        if not self.with_conflicts:
            return z
        conf = np.asarray(conf, float)
        return np.hstack([z, conf, conf * z])
    def fit(self, s, y, conf=None):
        self.lr.fit(self._X(s, conf), np.asarray(y, int)); return self
    def predict(self, s, conf=None):
        return self.lr.predict_proba(self._X(s, conf))[:, 1]

def cap(s, conflict, kappa):
    out = np.asarray(s, float).copy()
    out[conflict] = np.minimum(out[conflict], kappa)
    return out

# ---- entity level -----------------------------------------------------------------
def transitive_cleanup(li, ri, w, gate, max_iter=5):
    """Port of er.clustering.transitive_cleanup on integer node ids. Returns a keep-mask.
    For an open wedge b-a-c (b, c unlinked) the weaker edge is dropped only if its score < gate;
    violations are found per pass on one snapshot and applied together (order-independent)."""
    li, ri, w = list(map(int, li)), list(map(int, ri)), list(map(float, w))
    dropped = np.zeros(len(li), dtype=bool)
    for _ in range(max_iter):
        adj = {}
        for i in np.nonzero(~dropped)[0]:
            a, b, s = li[i], ri[i], w[i]
            adj.setdefault(a, {})[b] = (s, i)
            adj.setdefault(b, {})[a] = (s, i)
        props = set()
        for a, nb in adj.items():
            if len(nb) < 2:
                continue
            keys = sorted(nb)
            for j, x in enumerate(keys):
                ax = adj[x]
                for y_ in keys[j + 1:]:
                    if y_ in ax:
                        continue
                    (sx, ix), (sy, iy) = nb[x], nb[y_]
                    s, i = (sx, ix) if sx <= sy else (sy, iy)
                    if s < gate:
                        props.add(i)
        if not props:
            break
        dropped[list(props)] = True
    return ~dropped

def _pairs_and_max(labels):
    _, cnt = np.unique(labels, return_counts=True)
    return int((cnt * (cnt - 1) // 2).sum()), int(cnt.max()) if len(cnt) else 0

class SplitCtx:
    """Node indexing + gold clusters for one split's pairs."""
    def __init__(self, df, gold_map):
        nodes = pd.Index(pd.unique(np.r_[df.record_id_l.astype(str).values, df.record_id_r.astype(str).values]))
        self.li = nodes.get_indexer(df.record_id_l.astype(str))
        self.ri = nodes.get_indexer(df.record_id_r.astype(str))
        g = pd.Series(nodes, dtype=str).map(gold_map)
        g = g.where(g.notna(), "__nogold__" + pd.Series(nodes, dtype=str))
        self.gold = pd.factorize(g)[0]
        self.n = len(nodes)
        self.gold_pairs = _pairs_and_max(self.gold)[0]

    def entity(self, accept, w=None, gate=None):
        accept = np.asarray(accept, bool)
        li, ri = self.li[accept], self.ri[accept]
        if gate is not None and len(li):
            keep = transitive_cleanup(li, ri, np.asarray(w)[accept], gate)
            li, ri = li[keep], ri[keep]
        if len(li):
            g = coo_matrix((np.ones(len(li)), (li, ri)), shape=(self.n, self.n))
            lab = connected_components(g, directed=False)[1]
        else:
            lab = np.arange(self.n)
        key = lab.astype(np.int64) * (int(self.gold.max()) + 1) + self.gold
        tp = _pairs_and_max(key)[0]
        pp, mx = _pairs_and_max(lab)
        p = tp / pp if pp else 0.0
        r = tp / self.gold_pairs if self.gold_pairs else 0.0
        return {"entity_p": p, "entity_r": r, "entity_f1": 2 * p * r / (p + r) if p + r else 0.0, "max_cluster": mx}

def thr_grid(s, n=16):
    return np.unique(np.quantile(s, np.linspace(0.5, 0.995, n)))

def best_config(ctx, s, w, extra_thr=()):
    """Sweep (threshold, gate) and return {gate: (thr, f1)} plus the overall best (thr, gate, f1)."""
    grid = np.unique(np.r_[thr_grid(s), list(extra_thr)])
    res = {}
    for g in GATES:
        best = (-1.0, None)
        for t in grid:
            f = ctx.entity(s >= t, w, g)["entity_f1"]
            if f > best[0] + 1e-12 or (abs(f - best[0]) <= 1e-12 and (best[1] is None or t > best[1])):
                best = (f, t)
        res[g] = (best[1], best[0])
    g_best = max(GATES, key=lambda g: res[g][1])
    return res, (res[g_best][0], g_best, res[g_best][1])

def best_gate_at(ctx, s, w, t):
    return max(GATES, key=lambda g: ctx.entity(s >= t, w, g)["entity_f1"])

In [ ]:
# ---- one (corner, matcher, seed) run -> metric rows ---------------------------------
def review_eval(y, d, p, q, conf_full, ctx, W, rng, base):
    N = len(y)
    err_unc = np.where(d, 1 - p, p)
    pri = {"uncertainty": err_unc,
           "conflict_aware": np.where(d, 1 - q, q),
           "conflict_first": err_unc + 2.0 * (conf_full & d)}
    f1_none = prf(y, d)[2]
    rows = []
    for b in REVIEW_BUDGETS:
        k = int(round(b * N))
        preds = {}
        for name, pr in pri.items():
            idx = np.argsort(-pr, kind="mergesort")[:k]
            pred = d.copy(); pred[idx] = y[idx] == 1
            preds[name] = pred
        rand_f1 = []
        for r_ in range(10):
            idx = rng.choice(N, size=k, replace=False)
            pred = d.copy(); pred[idx] = y[idx] == 1
            rand_f1.append(prf(y, pred)[2])
            if r_ == 0:
                preds["random"] = pred
        for name, pred in preds.items():
            f1 = float(np.mean(rand_f1)) if name == "random" else prf(y, pred)[2]
            rows.append({**base, "budget": b, "policy": name, "f1": f1, "delta_f1": f1 - f1_none,
                         "errors_fixed": int((pred != d).sum()), **(ctx.entity(pred) if ctx is not None else {})})
        dlt = f1_weighted(y, preds["conflict_aware"], W) - f1_weighted(y, preds["uncertainty"], W)
        lo, hi = ci(dlt)
        rows.append({**base, "budget": b, "policy": "aware_minus_uncertainty",
                     "delta_f1": prf(y, preds["conflict_aware"])[2] - prf(y, preds["uncertainty"])[2],
                     "ci_lo": lo, "ci_hi": hi})
    return rows

def analyse_run(c, m, seed, P, C, gold_map, rng, has_gold=True):
    S = pd.read_csv(score_path(c, m, seed))
    D = P.merge(S, on="pid").merge(C, on="pid")
    V = D[D.split == "valid"]
    yv, sv = V.label.values.astype(int), V.score.values.astype(float)
    cv = V[["conf_full", "conf_sym"]].values.astype(bool)
    platt = Platt().fit(sv, yv)
    aware = Platt(with_conflicts=True).fit(sv, yv, cv)
    thr_f1, thr_pf = pick_threshold(yv, sv, "f1"), pick_threshold(yv, sv, "pf")
    qv = aware.predict(sv, cv)
    thr_q = pick_threshold(yv, qv, "f1")
    kappa, thr_cap = {}, {}
    for j, kind in enumerate(("full", "sym")):
        kappa[kind] = max(KAPPA_GRID, key=lambda k: (round(ap(yv, cap(sv, cv[:, j], k)), 12), k))
        thr_cap[kind] = pick_threshold(yv, cap(sv, cv[:, j], kappa[kind]), "f1")
    thr_ent = thr_best = gate_best = gate_pw = np.nan
    if has_gold:   # entity-level selection needs gold clusters (WDC Products only)
        vctx = SplitCtx(V, gold_map)
        pv = platt.predict(sv)
        per_gate, (thr_best, gate_best, _) = best_config(vctx, sv, pv, extra_thr=[thr_f1])
        thr_ent = per_gate[None][0]
        gate_pw = best_gate_at(vctx, sv, pv, thr_f1)

    rows, review_rows, rel = [], [], []
    for u in UNSEEN_OF.get(c, UNSEEN):
        T = D[D.split == f"test_{u}"]
        if T.empty:
            continue
        y, s = T.label.values.astype(int), T.score.values.astype(float)
        conf = T[["conf_full", "conf_sym"]].values.astype(bool)
        p, q = platt.predict(s), aware.predict(s, conf)
        W = rng.poisson(1.0, size=(N_BOOT, len(y))).astype(float)
        base = {"corner": c, "unseen": u, "matcher": m, "seed": seed}
        r = {**base, "n_test": len(y), "positives": int(y.sum()), "thr_f1": thr_f1, "thr_pf": thr_pf,
             "ap": ap(y, s), "ece_raw": ece(np.clip(s, 0, 1), y), "ece_cal": ece(p, y)}
        d = s >= thr_f1
        r["precision"], r["recall"], r["f1"] = prf(y, d)
        r["precision_pf"], r["recall_pf"], r["f1_pf"] = prf(y, s >= thr_pf)
        r["f1_at_0.5"] = prf(y, s >= 0.5)[2]
        r["f1_test_oracle"] = prf(y, s >= pick_threshold(y, s, "f1"))[2]
        hn = (T.is_hard_negative.values == 1) & (y == 0)
        r["hard_neg_false_merge"] = float(d[hn].mean()) if hn.any() else np.nan
        ap_boot_base = ap_weighted(y, s, W)
        f1_boot_base = f1_weighted(y, d, W)
        for j, kind in enumerate(("full", "sym")):
            cf = conf[:, j]
            blocked = d & cf
            r[f"conf_share_{kind}"] = float(cf.mean())
            r[f"veto_{kind}_blocked"] = int(blocked.sum())
            r[f"veto_{kind}_precision"] = float((y[blocked] == 0).mean()) if blocked.any() else np.nan
            r[f"veto_{kind}_true_blocked"] = int((y[blocked] == 1).sum())
            r[f"f1_veto_{kind}"] = prf(y, d & ~cf)[2]
            r[f"d_f1_shared_{kind}"] = r[f"f1_veto_{kind}"] - r["f1"]
            r[f"d_f1_shared_{kind}_lo"], r[f"d_f1_shared_{kind}_hi"] = ci(f1_weighted(y, d & ~cf, W) - f1_boot_base)
            sc = cap(s, cf, kappa[kind])
            r[f"kappa_{kind}"] = kappa[kind]
            r[f"d_ap_{kind}"] = ap(y, sc) - r["ap"]
            r[f"d_ap_{kind}_lo"], r[f"d_ap_{kind}_hi"] = ci(ap_weighted(y, sc, W) - ap_boot_base)
            r[f"f1_capped_own_{kind}"] = prf(y, sc >= thr_cap[kind])[2]
        r["ap_soft"] = ap(y, q)
        r["d_ap_soft"] = r["ap_soft"] - r["ap"]
        r["d_ap_soft_lo"], r["d_ap_soft_hi"] = ci(ap_weighted(y, q, W) - ap_boot_base)
        r["f1_soft"] = prf(y, q >= thr_q)[2]
        r["d_f1_soft"] = r["f1_soft"] - r["f1"]
        r["d_f1_soft_lo"], r["d_f1_soft_hi"] = ci(f1_weighted(y, q >= thr_q, W) - f1_boot_base)

        ctx = SplitCtx(T, gold_map) if has_gold else None
        if ctx is not None:
            configs = {"pw": ctx.entity(d), "ent": ctx.entity(s >= thr_ent),
                       "clean_pw": ctx.entity(d, p, gate_pw), "best": ctx.entity(s >= thr_best, p, gate_best),
                       "veto": ctx.entity(d & ~conf[:, 0]), "soft": ctx.entity(q >= thr_q)}
            _, (_, _, oracle_f1) = best_config(ctx, s, p, extra_thr=[thr_f1])
            for k_, v_ in configs.items():
                r[f"ent_{k_}_f1"] = v_["entity_f1"]
                r[f"ent_{k_}_max_cluster"] = v_["max_cluster"]
            r["ent_oracle_f1"] = oracle_f1
        r.update({"thr_ent": thr_ent, "thr_best": thr_best, "gate_best": gate_best, "gate_pw": gate_pw})
        rows.append(r)
        review_rows += review_eval(y, d, p, q, conf[:, 0], ctx, W, rng, base)
        if seed == seeds_for(m)[0]:
            rel.append(reliability(np.clip(s, 0, 1), y).assign(kind="raw", **base))
            rel.append(reliability(p, y).assign(kind="calibrated", **base))
    return rows, review_rows, rel

## 7. Run the analysis

In [ ]:
RUNS, REVIEW, REL = [], [], []
rng = np.random.default_rng(12345)
for c in KEYS:
    P = pd.read_csv(OUT / "data" / f"pairs_c{c}.csv.gz", dtype={"record_id_l": str, "record_id_r": str})
    C = pd.read_csv(OUT / "data" / f"conflicts_c{c}.csv.gz")
    recs = pd.read_csv(OUT / "data" / f"records_c{c}.csv.gz", dtype={"record_id": str, "gold_entity": str})
    gold_map = dict(zip(recs.record_id, recs.gold_entity))
    has_gold = bool(recs.gold_entity.notna().any())
    for m in MATCHERS:
        for seed in seeds_for(m):
            if not score_path(c, m, seed).exists():
                continue
            t0 = time.time()
            a, b, rel = analyse_run(c, m, seed, P, C, gold_map, rng, has_gold)
            RUNS += a; REVIEW += b; REL += rel
            print(f"analysed {c} {m:13s} seed {seed}: {time.time() - t0:.0f}s")

def dataset_of(c):
    return "WDC Products" if str(c).isdigit() else DATASET_NAME.get(c, str(c))

runs_all = pd.DataFrame(RUNS)
review_all = pd.DataFrame(REVIEW)
runs_all["dataset"] = runs_all.corner.map(dataset_of)
review_all["dataset"] = review_all.corner.map(dataset_of)
relt = pd.concat(REL, ignore_index=True) if REL else pd.DataFrame()
runs_all.to_csv(OUT / "tables" / "runs.csv", index=False)
review_all.to_csv(OUT / "tables" / "review.csv", index=False)
relt.to_csv(OUT / "tables" / "reliability.csv", index=False)
# Sections 8-10 report the nine-variant WDC grid; table_generalisation and fig_veto_vs_quality use all datasets.
runs = runs_all[runs_all.dataset == "WDC Products"].copy()
review = review_all[review_all.dataset == "WDC Products"].copy()
runs["corner"] = runs.corner.astype(int); review["corner"] = review.corner.astype(int)
print(len(runs_all), "run rows,", len(review_all), "review rows;", runs_all.dataset.value_counts().to_dict())

## 8. Paper tables (CSV + LaTeX)

In [ ]:
LABEL = {"tokenset": "Token-set similarity", "wordcooc": "Word co-occ. + SVM",
         "gbdt": "GBDT + attr. features", "biencoder": f"Bi-encoder ({BI_MODEL.split('/')[-1]})",
         "crossencoder": f"Cross-encoder ({CE_MODEL})", "deberta": f"Cross-encoder ({DEBERTA_MODEL.split('/')[-1]})",
         "llm": f"LLM zero-shot ({LLM_MODEL.split('/')[-1]})", "llm2": f"LLM zero-shot ({LLM2_MODEL.split('/')[-1]})"}
ORDER = [m for m in ["tokenset", "wordcooc", "gbdt", "biencoder", "llm2", "llm", "deberta", "crossencoder"]
         if m in set(runs.matcher)]

def variant_means(df, cols):
    """Average over seeds -> one value per (matcher, corner, unseen)."""
    return df.groupby(["matcher", "corner", "unseen"], as_index=False)[cols].mean()

def grid_stats(vm, col):
    """Across variants: mean, wins (>0), n, Wilcoxon p (two-sided)."""
    out = {}
    for m, g in vm.groupby("matcher"):
        x = g[col].dropna().values
        p = wilcoxon(x).pvalue if len(x) >= 5 and np.any(x != 0) else np.nan
        out[m] = {"mean": x.mean() if len(x) else np.nan, "wins": int((x > 0).sum()), "n": len(x), "p": p}
    return out

def sig_count(df, col):
    """Runs whose bootstrap CI excludes zero, as 'k+/k-/n'."""
    out = {}
    for m, g in df.groupby("matcher"):
        lo, hi = g[f"{col}_lo"], g[f"{col}_hi"]
        out[m] = f"{int((lo > 0).sum())}+/{int((hi < 0).sum())}-/{len(g)}"
    return out

def latex_table(df, path, caption, label, digits=3, wide=False):
    """IEEE-ready booktabs table; `wide` spans both columns. Headers may contain LaTeX math ($...$)."""
    def esc(x):
        if x is None:
            return "--"
        if isinstance(x, (float, np.floating)):
            return "--" if np.isnan(x) else f"{x:.{digits}f}".replace("-", "$-$")
        return str(x).replace("&", r"\&").replace("%", r"\%").replace("_", r"\_").replace("#", r"\#")
    cols = list(df.columns)
    env, width = ("table*", r"\textwidth") if wide else ("table", r"\columnwidth")
    numeric = [pd.api.types.is_numeric_dtype(df[c]) for c in cols]
    n_lead = next((i for i, isnum in enumerate(numeric) if isnum), len(cols)) or 1   # leading text columns
    lines = [rf"\begin{{{env}}}[t]", r"\centering", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             rf"\resizebox{{{width}}}{{!}}{{%", r"\begin{tabular}{" + "l" * n_lead + "r" * (len(cols) - n_lead) + "}",
             r"\toprule", " & ".join(c for c in cols) + r" \\", r"\midrule"]
    lines += [" & ".join(esc(v) for v in row) + r" \\" for row in df.itertuples(index=False)]
    lines += [r"\bottomrule", r"\end{tabular}}", rf"\end{{{env}}}"]
    pathlib.Path(path).write_text("\n".join(lines))

def save(df, name, caption, tex=None, wide=False):
    """CSV keeps every column; the LaTeX version keeps `tex` = {column: LaTeX header} in that order."""
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
    d = df[[c for c in tex if c in df.columns]].rename(columns=tex) if tex else df.rename(columns=lambda c: str(c).replace("_", r"\_").replace("%", r"\%"))
    for c in d.columns:
        if c.lower().startswith("budget"):
            d[c] = d[c].map(lambda b: f"{b:.0%}")
    latex_table(d, OUT / "latex" / f"{name}.tex", caption, f"tab:{name}", wide=wide)
    try:
        display(df)  # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string(index=False))

# ---- Table A: matching quality
cols = ["ap", "f1", "precision", "recall", "f1_pf", "f1_at_0.5", "ece_raw", "ece_cal", "hard_neg_false_merge"]
vm = variant_means(runs, cols)
tA = vm.groupby("matcher")[cols].mean().reindex(ORDER)
tA.insert(0, "F1 std (seeds)", runs.groupby(["matcher", "corner", "unseen"]).f1.std().groupby("matcher").mean().reindex(ORDER))
ref = vm[(vm.corner == 80) & (vm.unseen == 0)].set_index("matcher").reindex(ORDER)
tA.insert(0, "F1 80/0", ref["f1"])
tA.insert(1, "published 80/0", [PUBLISHED_F1_80_0.get(m) for m in ORDER])
tA = tA.reset_index().rename(columns={"index": "matcher"})
tA["matcher"] = tA["matcher"].map(LABEL)
save(tA.rename(columns={"ap": "AP", "f1": "F1", "f1_pf": "F1 (P@R>=.5)", "f1_at_0.5": "F1@0.5",
                        "ece_raw": "ECE raw", "ece_cal": "ECE cal", "hard_neg_false_merge": "hard-neg FMR"}),
     "table_matching", "Matching quality on WDC Products, mean over the nine variants and all seeds. F1, P and R "
     "use the threshold that maximises validation F1; F1@0.5 uses the model's own decision boundary; ECE uses 10 "
     "equal-mass bins before (raw) and after Platt scaling on validation (cal); FMR is the false-merge rate on "
     "the benchmark's hard negatives.",
     tex={"matcher": "Matcher", "AP": "AP", "F1": "F1", "precision": "P", "recall": "R", "F1@0.5": "F1@0.5",
          "ECE raw": "ECE raw", "ECE cal": "ECE cal", "hard-neg FMR": "Hard-neg.\\ FMR"})

# ---- Protocol parity with the base paper, every variant. F1@0.5 is the models' own decision boundary
# (for word co-occurrence that is the SVM's decision function = 0), which is how Table 3 was produced.
par = []
for m, refs in PUBLISHED_F1.items():
    if m not in ORDER:
        continue
    ours = variant_means(runs[runs.matcher == m], ["f1_at_0.5"]).set_index(["corner", "unseen"])["f1_at_0.5"]
    par += [{"matcher": LABEL[m], "corner": c, "unseen": u, "ours F1@0.5": ours[(c, u)], "published": pub,
             "diff": ours[(c, u)] - pub} for (c, u), pub in refs.items() if (c, u) in ours.index]
tP = pd.DataFrame(par)
if not tP.empty:
    save(tP, "table_published_parity", "Our reproduction versus the published F1 of Peeters et al. (EDBT 2024), "
         "medium development set.")
    print(tP.groupby("matcher")["diff"].agg(mean_bias="mean", max_abs=lambda x: x.abs().max()).round(4))

# ---- Table C/E: constraint
cols = ["d_ap_full", "d_f1_shared_full", "veto_full_precision", "veto_full_blocked", "veto_full_true_blocked",
        "d_ap_sym", "d_f1_shared_sym", "veto_sym_precision", "d_ap_soft", "d_f1_soft", "ap"]
vm = variant_means(runs, cols)
rows_ = []
for m in ORDER:
    g = {k: grid_stats(vm, k)[m] for k in ("d_ap_full", "d_f1_shared_full", "d_ap_sym", "d_ap_soft", "d_f1_soft")}
    sc = {k: sig_count(runs, k)[m] for k in ("d_ap_full", "d_f1_shared_full", "d_ap_soft")}
    mm = vm[vm.matcher == m].mean(numeric_only=True)
    rows_.append({"matcher": LABEL[m], "base AP": mm["ap"],
                  "dAP veto": g["d_ap_full"]["mean"], "dAP veto wins": f"{g['d_ap_full']['wins']}/{g['d_ap_full']['n']}",
                  "dAP veto CI": sc["d_ap_full"], "dAP veto p": g["d_ap_full"]["p"],
                  "dF1 shared": g["d_f1_shared_full"]["mean"],
                  "dF1 shared wins": f"{g['d_f1_shared_full']['wins']}/{g['d_f1_shared_full']['n']}",
                  "veto prec.": mm["veto_full_precision"], "blocked": mm["veto_full_blocked"],
                  "true blocked": mm["veto_full_true_blocked"],
                  "dAP sym": g["d_ap_sym"]["mean"], "veto prec. sym": mm["veto_sym_precision"],
                  "dAP soft": g["d_ap_soft"]["mean"], "dAP soft wins": f"{g['d_ap_soft']['wins']}/{g['d_ap_soft']['n']}",
                  "dAP soft CI": sc["d_ap_soft"], "dF1 soft": g["d_f1_soft"]["mean"]})
save(pd.DataFrame(rows_), "table_constraint",
     "Conflict evidence as a hard veto versus as a soft recalibration feature (WDC Products, nine variants). "
     "$\\Delta$ = change versus the unconstrained matcher, averaged over seeds and variants; wins = variants "
     "with a positive change; $p$ = two-sided Wilcoxon test over variants. Veto precision = share of non-matches "
     "among pairs the matcher accepts at its validation-F1 threshold that the veto rejects.",
     tex={"matcher": "Matcher", "base AP": "AP", "dAP veto": "$\\Delta$AP veto", "dAP veto wins": "wins",
          "dAP veto p": "$p$", "dF1 shared": "$\\Delta$F1 veto", "dF1 shared wins": "wins",
          "veto prec.": "Veto prec.", "true blocked": "Matches blocked", "dAP soft": "$\\Delta$AP soft",
          "dAP soft wins": "wins", "dF1 soft": "$\\Delta$F1 soft"}, wide=True)

# ---- Table B: entity level
cols = ["ent_pw_f1", "ent_ent_f1", "ent_clean_pw_f1", "ent_best_f1", "ent_soft_f1", "ent_veto_f1", "ent_oracle_f1",
        "ent_pw_max_cluster", "ent_best_max_cluster"]
runs["d_ent_best"] = runs.ent_best_f1 - runs.ent_pw_f1
runs["d_ent_ent"] = runs.ent_ent_f1 - runs.ent_pw_f1
vm = variant_means(runs, cols + ["d_ent_best", "d_ent_ent"])
gb, ge = grid_stats(vm, "d_ent_best"), grid_stats(vm, "d_ent_ent")
tB = vm.groupby("matcher")[cols].mean().reindex(ORDER).reset_index()
tB["best>pw"] = [f"{gb[m]['wins']}/{gb[m]['n']}" for m in ORDER]
tB["p"] = [gb[m]["p"] for m in ORDER]
tB["ent>pw"] = [f"{ge[m]['wins']}/{ge[m]['n']}" for m in ORDER]
tB["matcher"] = tB["matcher"].map(LABEL)
save(tB.rename(columns={"ent_pw_f1": "pairwise thr", "ent_ent_f1": "entity thr", "ent_clean_pw_f1": "pw+cleanup",
                        "ent_best_f1": "best (val)", "ent_soft_f1": "soft", "ent_veto_f1": "veto",
                        "ent_oracle_f1": "oracle (test)", "ent_pw_max_cluster": "max cl. pw",
                        "ent_best_max_cluster": "max cl. best"}),
     "table_entity", "Entity-level F1 on the WDC Products test splits (mean over nine variants and seeds). Every "
     "configuration except the oracle is selected on validation; the oracle is tuned on test and is an upper "
     "bound only. Wins and $p$ compare `best (val)' with the pairwise threshold.",
     tex={"matcher": "Matcher", "pairwise thr": "Pairwise thr.", "entity thr": "Entity thr.",
          "pw+cleanup": "Pairwise + cleanup", "best (val)": "Best (val.)", "oracle (test)": "Oracle (test)",
          "best>pw": "wins", "p": "$p$", "max cl. pw": "Max cl.\\ pairwise", "max cl. best": "Max cl.\\ best"},
     wide=True)

# ---- Table F: review routing
rv = review[review.policy != "aware_minus_uncertainty"]
tF = (rv.groupby(["matcher", "corner", "unseen", "budget", "policy"], as_index=False)[["delta_f1", "entity_f1"]].mean()
        .groupby(["matcher", "budget", "policy"])["delta_f1"].mean().unstack("policy").reset_index())
dm = review[review.policy == "aware_minus_uncertainty"]
dmv = dm.groupby(["matcher", "corner", "unseen", "budget"], as_index=False).delta_f1.mean()
extra = []
for (m, b), g in dmv.groupby(["matcher", "budget"]):
    x = g.delta_f1.values
    s_ = dm[(dm.matcher == m) & (dm.budget == b)]
    extra.append({"matcher": m, "budget": b, "aware-unc": x.mean(), "wins": f"{int((x > 0).sum())}/{len(x)}",
                  "CI sig": f"{int((s_.ci_lo > 0).sum())}+/{int((s_.ci_hi < 0).sum())}-/{len(s_)}",
                  "p": wilcoxon(x).pvalue if len(x) >= 5 and np.any(x != 0) else np.nan})
tF = tF.merge(pd.DataFrame(extra), on=["matcher", "budget"])
tF["matcher"] = pd.Categorical(tF.matcher, ORDER)
tF = tF.sort_values(["matcher", "budget"])
tF["matcher"] = tF.matcher.map(LABEL)
save(tF, "table_review", "Review routing on WDC Products: F1 gain over no review when a reviewer corrects the "
     "given share of test pairs. The last three columns compare conflict-aware with uncertainty routing "
     "(mean difference, variants won, Wilcoxon $p$).",
     tex={"matcher": "Matcher", "budget": "Budget", "random": "Random", "uncertainty": "Uncertainty",
          "conflict_first": "Conflict-first", "conflict_aware": "Conflict-aware", "aware-unc": "$\\Delta$",
          "wins": "wins", "p": "$p$"}, wide=True)

# ---- Table G: robustness to unseen products
vm = variant_means(runs, ["f1", "ap"])
tG = vm.groupby(["matcher", "unseen"]).f1.mean().unstack("unseen").reindex(ORDER)
if 0 in tG.columns and 100 in tG.columns:
    tG["drop 0->100"] = tG[0] - tG[100]
tG = tG.reset_index(); tG.columns = [str(x) for x in tG.columns]
tG["matcher"] = tG.matcher.map(LABEL)
save(tG, "table_robustness", "F1 by share of unseen products in the WDC Products test split (mean over the "
     "three corner-case levels and seeds).",
     tex={"matcher": "Matcher", "0": "Seen", "50": "Half-seen", "100": "Unseen", "drop 0->100": "Drop"})

# ---- Table H: generalisation to a second benchmark family (pairwise claims only)
B_ = CRITERIA["budget"]
gen = runs_all.groupby(["dataset", "matcher"])[["ap", "f1", "d_ap_full", "d_f1_shared_full", "veto_full_precision",
                                                 "d_ap_soft"]].mean()
gen = gen.join(review_all[(review_all.policy == "aware_minus_uncertainty") & (review_all.budget == B_)]
               .groupby(["dataset", "matcher"]).delta_f1.mean().rename(f"aware-unc@{B_:.0%}")).reset_index()
gen["matcher"] = pd.Categorical(gen.matcher, ORDER)
gen = gen.sort_values(["dataset", "matcher"])
gen["matcher"] = gen.matcher.map(LABEL)
save(gen.rename(columns={"ap": "AP", "f1": "F1", "d_ap_full": "dAP veto", "d_f1_shared_full": "dF1 shared",
                         "veto_full_precision": "veto prec.", "d_ap_soft": "dAP soft"}),
     "table_generalisation", "Pairwise results on every benchmark (WDC Products = mean of nine variants). "
     f"The last column is conflict-aware minus uncertainty routing at a {B_:.0%} review budget.".replace("%", "\\%"),
     tex={"dataset": "Dataset", "matcher": "Matcher", "AP": "AP", "dAP veto": "$\\Delta$AP veto",
          "veto prec.": "Veto prec.", "dAP soft": "$\\Delta$AP soft", f"aware-unc@{B_:.0%}": "$\\Delta$ review"})

# ---- Dataset statistics
dsf = OUT / "data" / "dataset_stats.csv"
if dsf.exists():
    save(pd.read_csv(dsf), "table_datasets", "Benchmarks. WDC Products shares train and validation splits across "
         "the unseen levels of a corner-case level; Magellan benchmarks have one test split and no gold clusters.",
         tex={"dataset": "Dataset", "variant": "Variant", "train": "Train", "train_pos": "pos.", "valid": "Valid",
              "test": "Test", "test_pos": "pos.", "test_hard_neg": "Hard neg."})

# ---- Table I: cost (from the Stage A timing log)
tf = OUT / "logs" / "timing.csv"
if tf.exists():
    tim = pd.read_csv(tf)
    tim["ms_per_pair"] = 1000 * tim.seconds / tim.pairs_scored
    tI = tim.groupby(["matcher", "model", "hardware"], as_index=False).agg(
        runs=("seconds", "size"), mean_minutes=("seconds", lambda x: x.mean() / 60), ms_per_pair=("ms_per_pair", "mean"))
    save(tI, "table_cost", "Wall-clock cost per run (training plus scoring every validation and test pair).",
         tex={"matcher": "Matcher", "model": "Model", "hardware": "Hardware", "runs": "Runs",
              "mean_minutes": "Minutes", "ms_per_pair": "ms/pair"})

## 9. Figures

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8, "figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
COL = dict(zip(["tokenset", "wordcooc", "gbdt", "biencoder", "llm2", "llm", "deberta", "crossencoder"],
               ["#8c8c8c", "#e69f00", "#56b4e9", "#d55e00", "#882255", "#cc79a7", "#0072b2", "#009e73"]))

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / "figures" / f"{name}.{ext}", bbox_inches="tight")
    plt.show()

# D: veto precision versus base-matcher quality
pts = runs_all.dropna(subset=["veto_full_precision"])
MARK = dict(zip(sorted(pts.dataset.unique(), key=lambda d: d != "WDC Products"), "os^Dv"))
fig, ax = plt.subplots(figsize=(3.4, 2.6))
for m in ORDER:
    for dname, mk in MARK.items():
        g = pts[(pts.matcher == m) & (pts.dataset == dname)]
        ax.scatter(g.ap, g.veto_full_precision, s=12, alpha=0.75, color=COL[m], marker=mk,
                   label=LABEL[m] if dname == "WDC Products" or "WDC Products" not in MARK else None)
for dname, mk in MARK.items():
    ax.scatter([], [], marker=mk, color="k", s=12, label=dname)
if len(pts) >= 5:
    rho, pval = spearmanr(pts.ap, pts.veto_full_precision)
    k, b0 = np.polyfit(pts.ap, pts.veto_full_precision, 1)
    xs = np.linspace(pts.ap.min(), pts.ap.max(), 50)
    ax.plot(xs, k * xs + b0, "k--", lw=0.8)
    ax.set_title(f"Spearman rho = {rho:.2f} (p = {pval:.1e}, n = {len(pts)})", fontsize=7)
ax.axhline(0.5, color="grey", lw=0.5, ls=":")
ax.set_xlabel("base matcher AP (test)"); ax.set_ylabel("veto precision at base threshold")
ax.legend(fontsize=6, frameon=False)
savefig(fig, "fig_veto_vs_quality")

# F: review routing curves
rv_m = (review[review.policy.isin(["uncertainty", "conflict_aware", "conflict_first", "random"])]
        .groupby(["matcher", "corner", "unseen", "budget", "policy"], as_index=False).delta_f1.mean())
show = [m for m in ("gbdt", "biencoder", "crossencoder", "deberta", "llm") if m in ORDER] or ORDER
fig, axes = plt.subplots(1, len(show), figsize=(2.4 * len(show), 2.2), sharey=False, squeeze=False)
for ax, m in zip(axes[0], show):
    for pol, st in [("uncertainty", "-o"), ("conflict_aware", "-s"), ("conflict_first", "--^"), ("random", ":x")]:
        g = rv_m[(rv_m.matcher == m) & (rv_m.policy == pol)].groupby("budget").delta_f1
        ax.errorbar(g.mean().index * 100, g.mean().values, yerr=g.std().values, fmt=st, ms=3, lw=1, capsize=2, label=pol)
    ax.set_title(LABEL[m], fontsize=7); ax.set_xlabel("pairs reviewed (%)")
axes[0][0].set_ylabel("F1 gain over no review"); axes[0][-1].legend(fontsize=6, frameon=False)
savefig(fig, "fig_review")

# B: entity-level configurations
fig, ax = plt.subplots(figsize=(3.6, 2.4))
keys = [("ent_pw_f1", "pairwise thr"), ("ent_ent_f1", "entity thr (val)"), ("ent_best_f1", "best (val)"),
        ("ent_oracle_f1", "oracle (test)")]
vm = variant_means(runs, [k for k, _ in keys])
x = np.arange(len(ORDER)); wbar = 0.2
for i, (k, lab) in enumerate(keys):
    g = vm.groupby("matcher")[k]
    ax.bar(x + (i - 1.5) * wbar, g.mean().reindex(ORDER), wbar, yerr=g.std().reindex(ORDER), capsize=1.5,
           label=lab, alpha=0.4 if "oracle" in k else 1.0)
ax.set_xticks(x); ax.set_xticklabels([LABEL[m].split(" (")[0] for m in ORDER], rotation=20, ha="right", fontsize=6)
ax.set_ylabel("entity-level F1 (test)"); ax.legend(fontsize=6, frameon=False)
savefig(fig, "fig_entity")

# G: robustness
fig, ax = plt.subplots(figsize=(3.0, 2.2))
rob = variant_means(runs, ["f1"]).groupby(["matcher", "unseen"]).f1.mean()
for m in ORDER:
    ax.plot(rob[m].index, rob[m].values, "-o", ms=3, color=COL[m], label=LABEL[m])
ax.set_xlabel("unseen products in test (%)"); ax.set_ylabel("F1"); ax.legend(fontsize=6, frameon=False)
savefig(fig, "fig_robustness")

# Calibration: reliability at 80/0 (first available corner/unseen)
if not relt.empty:
    c0, u0 = (80, 0) if ((relt.corner == 80) & (relt.unseen == 0)).any() else (relt.corner.iloc[0], relt.unseen.iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=(5.2, 2.4), sharey=True)
    for ax, kind in zip(axes, ("raw", "calibrated")):
        for m in ORDER:
            g = relt[(relt.kind == kind) & (relt.matcher == m) & (relt.corner == c0) & (relt.unseen == u0)]
            ax.plot(g.mean_p, g.observed, "-o", ms=2.5, color=COL[m], label=LABEL[m])
        ax.plot([0, 1], [0, 1], "k:", lw=0.7); ax.set_title(f"{kind} scores ({c0}%/{u0}%)", fontsize=7)
        ax.set_xlabel("predicted probability")
    axes[0].set_ylabel("observed match rate"); axes[1].legend(fontsize=5.5, frameon=False)
    savefig(fig, "fig_reliability")

## 10. Auto-generated findings summary

Every sentence below is computed from `tables/runs.csv` and `tables/review.csv`. Paste the numbers, not
impressions. If a verdict says *not supported*, the paper must say so.

In [ ]:
def fmt_p(p):
    return "n/a" if p is None or (isinstance(p, float) and np.isnan(p)) else f"{p:.3g}"

lines = [f"# Findings: run `{RUN_NAME}`", "",
         f"Variants: corners {CORNERS} x unseen {UNSEEN}; seeds {SEEDS} (train resample {TRAIN_SUBSAMPLE}); "
         f"bootstrap {N_BOOT}. Cross-encoder `{CE_MODEL}` ({CE_EPOCHS} epochs); LLM `{LLM_MODEL}`.", ""]

vmA = variant_means(runs, ["ap", "f1", "ece_raw", "ece_cal"])
lines.append("## Matching")
for m in ORDER:
    g = vmA[vmA.matcher == m]
    lines.append(f"- {LABEL[m]}: AP {g.ap.mean():.3f}, F1 {g.f1.mean():.3f}, ECE raw {g.ece_raw.mean():.3f} -> "
                 f"calibrated {g.ece_cal.mean():.3f} ({len(g)} variants).")

lines += ["", "## B. Entity-level threshold chosen on validation"]
vmB = variant_means(runs, ["d_ent_best", "d_ent_ent", "ent_pw_f1", "ent_best_f1", "ent_oracle_f1"])
for m in ORDER:
    g = vmB[vmB.matcher == m]; s_ = grid_stats(vmB, "d_ent_best")[m]
    verdict = "supported" if s_["wins"] == s_["n"] or (s_["wins"] >= 0.75 * s_["n"] and s_["mean"] > 0) else "not supported"
    lines.append(f"- {LABEL[m]}: pairwise-threshold entity F1 {g.ent_pw_f1.mean():.3f} -> validation-chosen "
                 f"{g.ent_best_f1.mean():.3f} (delta {s_['mean']:+.3f}, better in {s_['wins']}/{s_['n']}, "
                 f"Wilcoxon p={fmt_p(s_['p'])}); oracle ceiling {g.ent_oracle_f1.mean():.3f}. **{verdict}**")

lines += ["", "## C/D. Hard veto"]
vmC = variant_means(runs, ["d_ap_full", "d_f1_shared_full", "veto_full_precision", "ap"])
for m in ORDER:
    g = vmC[vmC.matcher == m]; s1 = grid_stats(vmC, "d_ap_full")[m]; s2 = grid_stats(vmC, "d_f1_shared_full")[m]
    lines.append(f"- {LABEL[m]}: dAP {s1['mean']:+.4f} (positive in {s1['wins']}/{s1['n']}, p={fmt_p(s1['p'])}); "
                 f"dF1 at shared threshold {s2['mean']:+.4f} ({s2['wins']}/{s2['n']}); veto precision "
                 f"{g.veto_full_precision.mean():.3f}.")
if len(pts) >= 5:
    rho, pval = spearmanr(pts.ap, pts.veto_full_precision)
    lines.append(f"- Veto precision vs base AP across {len(pts)} runs: Spearman rho = {rho:.2f} (p = {pval:.2g}). "
                 + ("**Veto precision falls as matcher quality rises.**" if rho < 0 and pval < 0.05 else
                    "No significant monotone relation; do not claim one."))

lines += ["", "## E. Soft conflict recalibration"]
vmE = variant_means(runs, ["d_ap_soft", "d_f1_soft"])
for m in ORDER:
    s1 = grid_stats(vmE, "d_ap_soft")[m]; s2 = grid_stats(vmE, "d_f1_soft")[m]
    lines.append(f"- {LABEL[m]}: dAP {s1['mean']:+.4f} ({s1['wins']}/{s1['n']}, p={fmt_p(s1['p'])}), "
                 f"dF1 {s2['mean']:+.4f} ({s2['wins']}/{s2['n']}).")

lines += ["", "## F. Conflict-aware review routing (conflict-aware minus uncertainty, F1)"]
for _, r_ in tF.iterrows():
    lines.append(f"- {r_['matcher']} @ {r_['budget']:.0%}: {r_['aware-unc']:+.4f} (better in {r_['wins']}, "
                 f"CI {r_['CI sig']}, p={fmt_p(r_['p'])})")

lines += ["", "## G. Robustness (F1 drop from 0% to 100% unseen)"]
if "drop 0->100" in tG.columns:
    for _, r_ in tG.iterrows():
        lines.append(f"- {r_['matcher']}: {r_['drop 0->100']:+.3f}")

lines += ["", "## H. Second benchmark family"]
for _, r_ in gen[gen.dataset != "WDC Products"].iterrows():
    lines.append(f"- {r_['dataset']} / {r_['matcher']}: AP {r_['ap']:.3f}, dAP veto {r_['d_ap_full']:+.4f}, "
                 f"veto precision {r_['veto_full_precision']:.3f}, aware-unc@{B_:.0%} {r_[f'aware-unc@{B_:.0%}']:+.4f}")

summary = "\n".join(lines)
(OUT / "FINDINGS_AUTO.md").write_text(summary)
print(summary)

## 11. Paper-readiness check (criteria fixed in section 1, before any result was seen)

Each claim passes only if it meets the criteria in `CRITERIA`. The cell then prints the paper framing those
results support. **Write the paper the results support, not the one you hoped for.** Changing `CRITERIA` after
seeing results invalidates this check.

In [ ]:
K = CRITERIA
def holds(st, sign=+1):
    """Consistent across the grid: right sign in >= min_wins_frac of variants and Wilcoxon p < alpha."""
    if not st or st["n"] == 0:
        return False
    wins = st["wins"] if sign > 0 else st["n"] - st["wins"]
    return wins >= K["min_wins_frac"] * st["n"] and sign * st["mean"] > 0 and (st["p"] == st["p"]) and st["p"] < K["alpha"]

strong = "crossencoder" if "crossencoder" in ORDER else ORDER[-1]
checks = {}
gB = grid_stats(variant_means(runs, ["d_ent_best"]), "d_ent_best")
nB = sum(holds(gB.get(m)) for m in ORDER)
checks["B_entity_threshold"] = (holds(gB.get(strong)) and nB >= (len(ORDER) + 1) // 2,
                                f"holds for {nB}/{len(ORDER)} matchers; strongest ({strong}): {holds(gB.get(strong))}")
gC = grid_stats(variant_means(runs, ["d_ap_full"]), "d_ap_full")
checks["C_veto_hurts_strong_matcher"] = (holds(gC.get(strong), sign=-1),
                                         f"{strong}: dAP {gC[strong]['mean']:+.4f}, positive in {gC[strong]['wins']}/{gC[strong]['n']}, p={fmt_p(gC[strong]['p'])}")
if len(pts) >= 5:
    rho, pv_ = spearmanr(pts.ap, pts.veto_full_precision)
    per_ds = {d: spearmanr(g.ap, g.veto_full_precision)[0] for d, g in pts.groupby("dataset") if len(g) >= 4}
    checks["D_veto_precision_falls_with_quality"] = (rho < 0 and pv_ < K["rho_alpha"] and sum(v < 0 for v in per_ds.values()) >= max(1, len(per_ds) - 1),
                                                     f"rho={rho:.2f}, p={pv_:.2g}; per dataset {', '.join(f'{d}: {v:.2f}' for d, v in per_ds.items())}")
gE = grid_stats(variant_means(runs, ["d_ap_soft"]), "d_ap_soft")
nE = sum(holds(gE.get(m)) for m in ORDER)
checks["E_soft_constraint_helps"] = (nE >= (len(ORDER) + 1) // 2, f"holds for {nE}/{len(ORDER)} matchers")
dF = review[(review.policy == "aware_minus_uncertainty") & (review.budget == K["budget"])]
gF = grid_stats(dF.groupby(["matcher", "corner", "unseen"], as_index=False).delta_f1.mean(), "delta_f1")
ext = review_all[(review_all.dataset != "WDC Products") & (review_all.policy == "aware_minus_uncertainty")
                 & (review_all.budget == K["budget"]) & (review_all.matcher == strong)].groupby("dataset").delta_f1.mean()
checks["F_conflict_aware_review"] = (holds(gF.get(strong)) and (len(ext) == 0 or (ext > 0).sum() >= max(1, len(ext) - 1)),
                                     f"{strong} on WDC: {gF[strong]['mean']:+.4f} ({gF[strong]['wins']}/{gF[strong]['n']}, p={fmt_p(gF[strong]['p'])}); "
                                     f"other datasets: {', '.join(f'{d} {v:+.4f}' for d, v in ext.items()) or 'none'}")
ce_par = tP[tP.matcher == LABEL.get("crossencoder")] if not tP.empty else tP
if len(ce_par) and CE_MODEL == "roberta-base":
    bias, worst = ce_par["diff"].mean(), ce_par["diff"].abs().max()
    checks["P_protocol_parity"] = (abs(bias) <= 0.03 and worst <= 0.06,
                                   f"cross-encoder vs published RoBERTa over {len(ce_par)} variants: mean bias "
                                   f"{bias:+.4f}, max |diff| {worst:.4f} (pass: |bias| <= 0.03 and max <= 0.06)")
else:
    checks["P_protocol_parity"] = (None, "needs the roberta-base cross-encoder on WDC Products")

if "deberta" in ORDER and strong in gC and strong in gF:
    gC_db = gC.get("deberta"); gF_db = gF.get("deberta")
    same_c = holds(gC_db, sign=-1) == holds(gC[strong], sign=-1)
    same_f = holds(gF_db) == holds(gF[strong])
    checks["R_deberta_replicates"] = (same_c and same_f,
                                      f"DeBERTa agrees with RoBERTa on C: {same_c} (dAP {gC_db['mean']:+.4f}, "
                                      f"{gC_db['wins']}/{gC_db['n']}), on F: {same_f} ({gF_db['mean']:+.4f}, "
                                      f"{gF_db['wins']}/{gF_db['n']})")
ok = {k: v[0] for k, v in checks.items()}
report = ["# Paper-readiness check", "", f"Criteria: {K}", ""]
report += [f"- [{'PASS' if v[0] else ('----' if v[0] is None else 'FAIL')}] {k}: {v[1]}" for k, v in checks.items()]
B, C_, D_, E_, F_ = (ok.get(k) for k in ("B_entity_threshold", "C_veto_hurts_strong_matcher",
                                         "D_veto_precision_falls_with_quality", "E_soft_constraint_helps",
                                         "F_conflict_aware_review"))
report += ["", "## Framing the results support"]
if B and D_ and F_:
    report.append("**Method + analysis paper (strongest).** Three claims: (1) choose the operating point for "
                  "entities on validation; (2) a hard constraint's precision falls as matcher quality rises; "
                  "(3) the same conflict evidence is valuable as a review-routing signal. Target: IEEE BigData main track.")
elif B and (D_ or C_):
    report.append("**Analysis paper with one actionable method.** Lead with the validation-selected entity "
                  "threshold, support it with the constraint-vs-quality finding. Target: IEEE BigData or an IEEE ICDE workshop.")
elif D_ or C_:
    report.append("**Negative result with a mechanism.** Frame as an empirical study of when hard constraints "
                  "help. Target: an IEEE workshop. Strengthen with a second positive result before a main track.")
else:
    report.append("**Not ready.** No central claim met its pre-set criterion. Do not submit; discuss with your "
                  "guide which claim to redesign rather than relaxing the criteria.")
if E_:
    report.append("- Add: conflict evidence helps as a soft signal (supporting result).")
if C_ is False:
    report.append("- Do NOT claim the veto harms strong matchers: it did not hold across the grid.")
report += ["", "Not covered by this notebook and still required: rewrite `paper/main.tex` around the passing "
           "claims, related work on LLM matchers, page limit, venue call, departmental publication clearance."]
readiness = "\n".join(report)
(OUT / "PAPER_READINESS.md").write_text(readiness)
print(readiness)

## 12. Numbers for the paper (`latex/numbers.tex`)

`paper/main.tex` never contains a typed-in result. It reads every number from this file through `\R{key}`, and
chooses supported or unsupported wording for each claim from the readiness flags (`\IfClaim{B}{...}{...}`).
Keys look like `CE.entbest` (matcher code + metric). Matcher codes: TS, WC, GB, LLM, CE.

In [ ]:
CODE = {"tokenset": "TS", "wordcooc": "WC", "gbdt": "GB", "biencoder": "BI", "llm": "LLM", "llm2": "LLMb",
        "deberta": "DB", "crossencoder": "CE"}
NUM = {}

def f3(x):
    return "--" if x is None or x != x else f"{x:.3f}"

def sg(x, d=3):   # \ensuremath keeps values valid both in running text and inside $...$
    return "--" if x is None or x != x else ("\\ensuremath{+}" if x >= 0 else "\\ensuremath{-}") + f"{abs(x):.{d}f}"

def pv(x):
    return "n/a" if x is None or x != x else ("\\ensuremath{<}0.001" if x < 0.001 else f"{x:.3f}")

def tex_text(s):
    return str(s).replace("\\", "/").replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

def put(key, fn):
    try:
        NUM[key] = fn()
    except Exception as exc:  # noqa: BLE001 - a missing number must not stop the file; it shows red in the PDF
        print(f"  (skipped {key}: {type(exc).__name__}: {exc})")

num_cols = ["ap", "f1", "f1_at_0.5", "precision", "recall", "ece_raw", "ece_cal", "hard_neg_false_merge",
            "d_ap_full", "d_f1_shared_full", "veto_full_precision", "veto_full_blocked", "veto_full_true_blocked",
            "conf_share_full", "d_ap_sym", "veto_sym_precision", "d_ap_soft", "d_f1_soft",
            "ent_pw_f1", "ent_ent_f1", "ent_clean_pw_f1", "ent_best_f1", "ent_oracle_f1",
            "ent_pw_max_cluster", "ent_best_max_cluster", "d_ent_best", "d_ent_ent"]
vmx = variant_means(runs, [c for c in num_cols if c in runs.columns])
GS = {c: grid_stats(vmx, c) for c in ("d_ap_full", "d_f1_shared_full", "d_ap_soft", "d_f1_soft", "d_ent_best", "d_ent_ent")}
rvb = review[review.budget == B_]
GA = grid_stats(rvb[rvb.policy == "aware_minus_uncertainty"]
                .groupby(["matcher", "corner", "unseen"], as_index=False).delta_f1.mean(), "delta_f1")
pol = rvb[rvb.policy != "aware_minus_uncertainty"].groupby(["matcher", "policy"]).delta_f1.mean()
rob = variant_means(runs, ["f1"]).groupby(["matcher", "unseen"]).f1.mean()
ext_all = runs_all[runs_all.dataset != "WDC Products"]
ext_rev = review_all[(review_all.dataset != "WDC Products") & (review_all.policy == "aware_minus_uncertainty")
                     & (review_all.budget == B_)]
tim = pd.read_csv(OUT / "logs" / "timing.csv") if (OUT / "logs" / "timing.csv").exists() else pd.DataFrame()

for m in ORDER:
    k, g = CODE[m], vmx[vmx.matcher == m]
    for key, col, fmt in [("ap", "ap", f3), ("fone", "f1", f3), ("fonehalf", "f1_at_0.5", f3), ("prec", "precision", f3),
                          ("rec", "recall", f3), ("eceraw", "ece_raw", f3), ("ececal", "ece_cal", f3),
                          ("fmr", "hard_neg_false_merge", f3), ("dapveto", "d_ap_full", sg),
                          ("dfoneshared", "d_f1_shared_full", sg), ("vetoprec", "veto_full_precision", f3),
                          ("vetoprecsym", "veto_sym_precision", f3), ("dapsym", "d_ap_sym", sg),
                          ("confshare", "conf_share_full", f3), ("dapsoft", "d_ap_soft", sg), ("dfonesoft", "d_f1_soft", sg),
                          ("entpw", "ent_pw_f1", f3), ("entent", "ent_ent_f1", f3), ("entclean", "ent_clean_pw_f1", f3),
                          ("entbest", "ent_best_f1", f3), ("entoracle", "ent_oracle_f1", f3),
                          ("dentbest", "d_ent_best", sg), ("dentent", "d_ent_ent", sg)]:
        put(f"{k}.{key}", lambda col=col, fmt=fmt: fmt(g[col].mean()))
    put(f"{k}.blocked", lambda: f"{g.veto_full_blocked.mean():.0f}")
    put(f"{k}.trueblocked", lambda: f"{g.veto_full_true_blocked.mean():.0f}")
    put(f"{k}.maxclpw", lambda: f"{g.ent_pw_max_cluster.mean():.0f}")
    put(f"{k}.maxclbest", lambda: f"{g.ent_best_max_cluster.mean():.0f}")
    for key, col in [("dapveto", "d_ap_full"), ("dfoneshared", "d_f1_shared_full"), ("dapsoft", "d_ap_soft"),
                     ("dfonesoft", "d_f1_soft"), ("entbest", "d_ent_best"), ("entent", "d_ent_ent")]:
        put(f"{k}.{key}wins", lambda col=col: f"{GS[col][m]['wins']}/{GS[col][m]['n']}")
        put(f"{k}.{key}p", lambda col=col: pv(GS[col][m]["p"]))
    put(f"{k}.awareunc", lambda: sg(GA[m]["mean"], 4))
    put(f"{k}.awareuncwins", lambda: f"{GA[m]['wins']}/{GA[m]['n']}")
    put(f"{k}.awareuncp", lambda: pv(GA[m]["p"]))
    put(f"{k}.revunc", lambda: sg(pol[(m, "uncertainty")]))
    put(f"{k}.revaware", lambda: sg(pol[(m, "conflict_aware")]))
    put(f"{k}.revrandom", lambda: sg(pol[(m, "random")]))
    put(f"{k}.unseendrop", lambda: f3(rob[(m, 0)] - rob[(m, 100)]))
    put(f"{k}.fonehalfeighty", lambda: f3(vmx[(vmx.matcher == m) & (vmx.corner == 80) & (vmx.unseen == 0)]["f1_at_0.5"].mean()))
    ge = ext_all[ext_all.matcher == m]
    put(f"{k}.genap", lambda: f3(ge.ap.mean()))
    put(f"{k}.genvetoprec", lambda: f3(ge.veto_full_precision.mean()))
    put(f"{k}.gendapveto", lambda: sg(ge.d_ap_full.mean()))
    put(f"{k}.gendapsoft", lambda: sg(ge.d_ap_soft.mean()))
    gr = ext_rev[ext_rev.matcher == m].groupby("dataset").delta_f1.mean()
    put(f"{k}.genawareunc", lambda: sg(gr.mean(), 4))
    put(f"{k}.genawarewins", lambda: f"{int((gr > 0).sum())}/{len(gr)}")
    if not tim.empty and (tim.matcher == m).any():
        tm = tim[tim.matcher == m]
        put(f"{k}.minutes", lambda: f"{tm.seconds.mean() / 60:.1f}")
        put(f"{k}.mspair", lambda: f"{1000 * (tm.seconds / tm.pairs_scored).mean():.2f}")
    if not tP.empty and (tP.matcher == LABEL[m]).any():
        tp_ = tP[tP.matcher == LABEL[m]]
        put(f"{k}.paritybias", lambda: sg(tp_["diff"].mean()))
        put(f"{k}.paritymax", lambda: f3(tp_["diff"].abs().max()))
    if m in PUBLISHED_F1:
        put(f"{k}.pubeighty", lambda: f3(PUBLISHED_F1[m][(80, 0)]))

if len(pts) >= 5:
    rho, pv_ = spearmanr(pts.ap, pts.veto_full_precision)
    put("rho", lambda: f"{rho:.2f}".replace("-", "\\ensuremath{-}"))
    def sci(x):
        if x != x:
            return "n/a"
        if x >= 0.001:
            return f"{x:.3f}"
        mant, exp = f"{x:.1e}".split("e")
        return "\\ensuremath{" + mant + "\\times10^{" + str(int(exp)) + "}}"
    put("rhop", lambda: sci(pv_))
    put("npts", lambda: str(len(pts)))
put("nvar", lambda: str(runs.groupby(["corner", "unseen"]).ngroups))
put("nseeds", lambda: str(len(SEEDS)))
put("nboot", lambda: f"{N_BOOT:,}")
put("trainsub", lambda: f"{TRAIN_SUBSAMPLE:.0%}".replace("%", r"\%"))
put("budget", lambda: f"{B_:.0%}".replace("%", r"\%"))
put("budgets", lambda: ", ".join(f"{b:.0%}" for b in REVIEW_BUDGETS).replace("%", r"\%"))
put("cemodel", lambda: tex_text(CE_MODEL))
put("ceepochs", lambda: str(CE_EPOCHS))
put("llmmodel", lambda: tex_text(LLM_MODEL.split("/")[-1]))
put("nextra", lambda: str(len(EXTRA_DATASETS)))
put("extranames", lambda: ", ".join(DATASET_NAME[k] for k in EXTRA_DATASETS))
put("minwins", lambda: f"{int(round(CRITERIA['min_wins_frac'] * 9))}/9")
put("alpha", lambda: f"{CRITERIA['alpha']}")
put("gpu", lambda: tex_text(torch.cuda.get_device_name(0)) if HAS_GPU else "CPU")
for name, flag in ok.items():
    if flag is not None:
        NUM[f"flag.{name.split('_')[0]}"] = "1" if flag else "0"

tex = ["% Generated by notebooks/er_full_pipeline_colab.ipynb (run " + tex_text(RUN_NAME) + "). Do not edit."]
tex += ["\\expandafter\\def\\csname res:" + k + "\\endcsname{" + v + "}" for k, v in sorted(NUM.items())]
(OUT / "latex" / "numbers.tex").write_text("\n".join(tex) + "\n")
print(f"wrote {len(NUM)} numbers -> {OUT / 'latex' / 'numbers.tex'}")
for fig in (OUT / "figures").glob("*.pdf"):
    shutil.copy(fig, OUT / "latex" / fig.name)   # latex/ is now everything paper/generated/ needs

## 13. Package and download everything

The zip contains `scores/` (raw evidence), `data/`, `tables/`, `latex/` (drop into `paper/tables/`),
`figures/` and `FINDINGS_AUTO.md`. To regenerate tables without a GPU (e.g. after changing a definition), run
section 1, set `OUT` to an unzipped copy, run the first cell of section 5 (helpers), then sections 6-10.

In [ ]:
meta = {"run": RUN_NAME, "quick": QUICK, "corners": CORNERS, "unseen": UNSEEN, "seeds": SEEDS,
        "train_subsample": TRAIN_SUBSAMPLE, "matchers": MATCHERS, "ce": [CE_MODEL, CE_EPOCHS, CE_BATCH, CE_LR, CE_MAX_LEN],
        "llm": LLM_MODEL, "n_boot": N_BOOT, "kappa_grid": KAPPA_GRID, "gates": [str(g) for g in GATES],
        "budgets": REVIEW_BUDGETS, "torch": torch.__version__, "python": sys.version.split()[0],
        "gpu": torch.cuda.get_device_name(0) if HAS_GPU else None,
        "repo_commit": subprocess.run(["git", "-C", str(PROJ), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()}
(OUT / "run_meta.json").write_text(json.dumps(meta, indent=2))
zip_path = shutil.make_archive(str(BASE / f"er_results_{RUN_NAME}"), "zip", OUT)
print("zip:", zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
if IN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as exc:  # noqa: BLE001
        print("download it from the Files pane:", exc)